Scratch Jupyter Noteboook

In [1]:
import numpy as np
from numpy.typing import NDArray
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import tqdm
import nonlinear_approximator as na
from scipy.stats import qmc
import torchvision
import tempfile
import numba
import torch

%matplotlib inline

In [2]:
# 
def xor(v) -> int:
    x, y = v[0], v[1]
    
    if (x < 0 and y > 0) or (x > 0 and y < 0):
        return 0 # return 0 if in 2nd or 4th quadrant 
    else:
        return 1 # return 1 if in 1st or 3rd quadrant


def tent_torch(x, mu=1.99):
    return mu * torch.minimum(x, 1 - x)

def compute_activations_torch(neurons, input_x, depth):
    """
    neurons: (dim_input, width)
    input_x: (num_samples, dim_input)

    Returns:
        (num_samples, depth * width)
    """
    num_samples, dim_input = input_x.shape
    width = neurons.shape[1]

    # allocate final output directly
    out = torch.empty(
        (num_samples, depth * width),
        device=input_x.device,
        dtype=input_x.dtype
    )

    # ---- layer 0 ----
    x = 0.5 * ((input_x + 1.0) @ neurons) / dim_input
    out[:, 0:width] = x

    # ---- subsequent layers ----
    for l in range(1, depth):
        x = tent_torch(x)
        out[:, l*width:(l+1)*width] = x

    return out

def gen_neurons(dimension, number):
    sampler = qmc.LatinHypercube(d=dimension)
    sample = sampler.random(n=number)
    return sample


def to_numpy_arr(img) -> NDArray[np.floating]:
    arr = np.asarray(img).flatten()
    arr = arr / 255 # 0 --> 1
    arr = arr - .5  # -.5 --> .5
    arr = 2 * arr   # -1 --> 1
    return arr



In [3]:
train_data = torchvision.datasets.MNIST(root='./data', download=True, train=True, transform=to_numpy_arr)
test_data = torchvision.datasets.MNIST(root='./data', download=True, train=False, transform=to_numpy_arr)
imgs_train, labels_train = zip(*train_data)

imgs_train = torch.from_numpy(np.array(imgs_train, dtype=np.float32))
labels_train = torch.from_numpy(np.array(labels_train, dtype=np.float32))

imgs_test, labels_test = zip(*test_data)
imgs_test = torch.from_numpy(np.array(imgs_test, dtype=np.float32))
labels_test = torch.from_numpy(np.array(labels_test, dtype=np.float32))

config = na.params.RegressionParams(
    width=28**2,
    depth=100,
    input_dimension=28**2,
    transform_type=na.activations.TransformType.TENT,
    transform_params=na.params.TentParams(mu=1.99),
    output_dimension=1,
)
neurons = gen_neurons(config.input_dimension, config.width).T

In [ ]:


decoders = {}
dim_in = config.depth * config.width
dim_out = config.output_dimension

activations_train = compute_activations_torch(
    neurons, input_x=imgs_train, depth=config.depth,
)

for idx in tqdm.tqdm(range(10)):
    labels_idx_train = torch.from_numpy(np.where(labels_train == idx, 1, 0).astype(np.float32))
    labels_idx_test = torch.from_numpy(np.where(labels_test == idx, 1, 0).astype(np.float32))
    decoder = na.training.BlockLowRankRLS(dim_in, dim_out, delta=1e-6, max_rank=config.depth * config.width)
    decoder.fit(
        activations_train.to(device='cuda'), 
        labels_idx_train.to(device='cuda'),
        batch_size=2048
        )
        
    decoders[idx] = decoder.W
    
np.savez('decoders', decoders)



/tmp/ipykernel_11607/134907966.py:33: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x = 0.5 * ((input_x + 1.0) @ neurons) / dim_input
  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
accs_train = torch.zeros([10,], device=neurons.device)
accs_test = torch.zeros([10,], device=neurons.device)

activations_train = compute_activations_torch(
    neurons, input_x=imgs_train, depth=config.depth,
)

activations_test = compute_activations_torch(
    neurons, input_x=imgs_test, depth=config.depth,
)

for idx in range(10):
    decoder = na.training.BlockLowRankRLS(dim_in, dim_out, delta=1e-2, max_rank=config.depth * config.width)
    decoder.W = decoders[idx]
    
    # predictions (stay on torch)
    preds_train = decoder.predict(activations_train)
    preds_test = decoder.predict(activations_test)

    # threshold in torch (no numpy)
    preds_train = (preds_train >= 0.5).int()
    preds_test = (preds_test >= 0.5).int()

    # binary labels in torch
    labels_idx_train = (labels_train.to(preds_train.device) == idx).int()
    labels_idx_test = (labels_test.to(preds_test.device) == idx).int()

    # accuracy (vectorized, no Python loops)
    accs_train[idx] = (preds_train.eq(labels_idx_train).float().mean()).item()
    accs_test[idx] = (preds_test.eq(labels_idx_test).float().mean()).item()

In [ ]:
plt.scatter(range(len(accs_train)), accs_train, label='train')
plt.scatter(range(len(accs_test)), accs_test, label='test')
plt.legend()
plt.xticks(ticks=range(10), labels=[f"{idx}" for idx in range(10)])
plt.ylim([0, 1])
plt.title('Binary-Classifier MNIST Dataset Accuracy (Is it X or  not X)')
plt.show()


In [ ]:

preds_train = np.zeros((len(imgs_train), 10))
preds_test =  np.zeros((len(imgs_test), 10))
for idx in range(10):
    preds_train[:, idx] = activations_train @ decoders[idx] + intercepts[idx]    
    preds_test[:, idx] = activations_test @ decoders[idx] + intercepts[idx]



In [ ]:
p_train_labels = preds_train.argmax(axis=-1)
p_test_labels = preds_test.argmax(axis=-1)
accs_train = p_train_labels == labels_train
accs_test = p_test_labels == labels_test
accs_train.mean(), accs_test.mean()